# 🤖 Best Model Choose Agent — Using LangChain & LangGraph

---

## What will we build?

In this notebook, we will build an **AI Agent** that acts like an automated Data Scientist.  
It will do the following things **step-by-step on its own**:

1. **EDA (Exploratory Data Analysis)** — Understand the dataset  
2. **Data Preprocessing** — Clean and prepare the data for ML  
3. **Model Building** — Train 3 different Machine Learning models  
4. **Evaluation** — Compare all models and pick the best one  

---

## What is LangChain?

**LangChain** is a Python framework that helps us connect **Large Language Models (LLMs)** (like ChatGPT, Llama, Gemini) with our own data, tools, and code.  
Think of it as a **bridge** between an AI brain and the real world.

## What is LangGraph?

**LangGraph** is a library built on top of LangChain that lets us define **workflows as graphs**.  
A graph is made up of:
- **Nodes** — These are the individual steps (like "Do EDA", "Preprocess", "Train Model").
- **Edges** — These connect the nodes and define the **order** in which things happen.

**Why use LangGraph?**  
Instead of writing one giant function, we break our pipeline into small, clean, reusable steps.  
LangGraph manages the flow of data between these steps automatically.

---

## Our Dataset: Bank Marketing Dataset

This dataset comes from a Portuguese bank and contains information about marketing phone calls.  
The goal is to predict whether a client will **subscribe to a term deposit** (column `y`: `yes` or `no`).

| Column | Description |
|--------|-------------|
| `age` | Age of the client |
| `job` | Type of job (admin, technician, etc.) |
| `marital` | Marital status |
| `education` | Education level |
| `default` | Has credit in default? |
| `housing` | Has housing loan? |
| `loan` | Has personal loan? |
| `duration` | Last call duration in seconds |
| `campaign` | Number of contacts during campaign |
| `y` | **Target** — Did the client subscribe? (yes/no) |

---
## Step 1: Install Required Libraries

We need a few Python packages:
- **`langgraph`** — To create our agent workflow graph
- **`langchain-groq`** — To use the Groq LLM (Llama 3.3) for generating a final AI report
- **`langchain-core`** — Core LangChain utilities
- **`python-dotenv`** — To load API keys securely from a `.env` file
- **`pandas`** — For data manipulation (like working with Excel but in Python)
- **`scikit-learn`** — For Machine Learning models and preprocessing

Run the cell below to install them all:

In [ ]:
!pip install langgraph langchain-groq langchain-core python-dotenv pandas scikit-learn -q

---
## Step 2: Import All Libraries

Let's import everything we need:

- **`os`** — To work with file paths and environment variables.
- **`TypedDict, Dict, Any`** — These are Python type hints. We use `TypedDict` to define the shape of our LangGraph state (what data flows between nodes).
- **`pandas`** — Our main data manipulation library.
- **`sklearn` modules** — For splitting data, encoding labels, scaling features, building models, and measuring accuracy.
- **`StateGraph, START, END`** — From LangGraph, these let us build and control our workflow graph.
- **`ChatGroq`** — The LLM wrapper that connects to the Groq API to use Llama 3.3.

In [ ]:
import os
from typing import TypedDict, Dict, Any

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

print("All libraries imported successfully! ✅")

---
## Step 3: Set Up the Groq API Key

### What is an API Key?
An API key is like a **password** that lets your code talk to an external AI service (in our case, Groq).  
Without this key, the Groq LLM won't respond to our requests.

### How to get a Groq API Key?
1. Go to [console.groq.com](https://console.groq.com)
2. Sign up / Log in
3. Navigate to **API Keys** and create a new key
4. Copy the key and paste it below

### ⚠️ Important:
Replace `"YOUR_GROQ_API_KEY_HERE"` with your actual key. Never share your API key publicly!

In [ ]:
# Option 1: Set API key directly (for Google Colab)
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"

# Option 2: If running locally with a .env file, uncomment the lines below:
# from dotenv import load_dotenv
# load_dotenv()

print("API Key set! ✅")

---
## Step 4: Upload & Load the Dataset

### For Google Colab:
Click the **files icon** on the left sidebar → Upload `bank-additional-full.csv`  
OR run the upload cell below.

### For Local:
Make sure the CSV file is in the same folder as this notebook.

### What is a CSV file?
CSV stands for **Comma Separated Values**. It's like a simple Excel file where each row is a record and columns are separated by commas (or, in our case, semicolons `;`).

In [ ]:
# Uncomment below if you are on Google Colab and want to upload the file
# from google.colab import files
# uploaded = files.upload()

# Define the path to the dataset
DATA_PATH = "bank-additional-full.csv"

# Quick preview — let's make sure the file loads correctly
df_preview = pd.read_csv(DATA_PATH, sep=';')
print(f"Dataset loaded with {df_preview.shape[0]} rows and {df_preview.shape[1]} columns")
df_preview.head()

---
## Step 5: Define the Agent State (using TypedDict)

### What is State?
In LangGraph, **State** is a dictionary that carries data between nodes.  
Think of it like a **shared notebook** that each team member (node) can read from and write to.

### What is TypedDict?
`TypedDict` is a Python feature that lets us define **what keys** our dictionary should have and **what type** of data each key holds.  
This helps us stay organized and catch bugs early.

Our state will carry:
| Key | Type | Purpose |
|-----|------|--------|
| `data_path` | `str` | Path to the CSV file |
| `eda_summary` | `str` | Summary text from EDA |
| `X_train`, `X_test` | `Any` | Training and testing feature matrices |
| `y_train`, `y_test` | `Any` | Training and testing labels |
| `model_accuracies` | `Dict[str, float]` | Each model's name → accuracy |
| `best_model_name` | `str` | The winning model |
| `final_report` | `str` | AI-generated summary report |

In [ ]:
class AgentState(TypedDict):
    data_path: str       # Path to CSV file
    eda_summary: str     # EDA output text
    X_train: Any         # Training features (numpy array)
    X_test: Any          # Testing features (numpy array)
    y_train: Any         # Training labels
    y_test: Any          # Testing labels
    model_accuracies: Dict[str, float]  # {"Model Name": accuracy_score}
    best_model_name: str   # Name of the best model
    final_report: str      # AI-generated final report

print("Agent State defined! ✅")

---
## Step 6: Create Node 1 — EDA (Exploratory Data Analysis)

### What is EDA?
Before building any ML model, we need to **understand our data**. EDA answers questions like:
- How many rows and columns do we have?
- What does the target variable look like? Is it balanced?
- Are there any missing values?

### What happens in this node?
1. We load the CSV file using `pd.read_csv()`
2. We check the **shape** (rows × columns)
3. We look at the **target distribution** (`y` column → how many "yes" vs "no")
4. We count **missing values**
5. We store this info in the state so the next node can use it

In [ ]:
def perform_eda(state: AgentState):
    """
    NODE 1: Exploratory Data Analysis
    
    This function loads the dataset and gathers basic statistics.
    It's the first step in any Data Science project.
    """
    print("="*60)
    print("🔍 [NODE 1] Performing EDA...")
    print("="*60)
    
    # Load the dataset — our file uses semicolons (;) as separators
    data_path = state["data_path"]
    df = pd.read_csv(data_path, sep=';')
    
    # --- Basic Info ---
    shape_info = f"📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns"
    print(shape_info)
    
    # --- Column Data Types ---
    print("\n📋 Column Data Types:")
    print(df.dtypes)
    
    # --- Target Variable Distribution ---
    # This tells us if our data is "imbalanced" (way more 'no' than 'yes')
    target_dist = df['y'].value_counts(normalize=True) * 100
    target_info = f"\n🎯 Target variable 'y' distribution (%):\n{target_dist}"
    print(target_info)
    
    # --- Missing Values ---
    missing_total = df.isnull().sum().sum()
    missing_info = f"\n❓ Total Missing Values: {missing_total}"
    print(missing_info)
    
    # Combine into a summary string to pass forward in state
    eda_summary = f"{shape_info}\n{target_info}\n{missing_info}"
    
    print("\n✅ EDA Complete!")
    
    # Return only the keys we want to UPDATE in the state
    return {"eda_summary": eda_summary}

print("Node 1 (EDA) defined! ✅")

---
## Step 7: Create Node 2 — Data Preprocessing

### What is Preprocessing?
Machine Learning models **cannot understand text** like "married" or "admin.".  
We need to convert everything into **numbers**. This process is called **preprocessing**.

### Steps we perform:

| Step | What it does | Why? |
|------|-------------|------|
| **Drop NAs** | Remove rows with missing data | Incomplete data can confuse models |
| **Separate X and y** | Split features (inputs) from target (output) | Models need inputs and outputs separately |
| **Label Encode y** | Convert `yes`→1, `no`→0 | Models need numbers, not text |
| **One-Hot Encode X** | Convert categorical columns to binary columns | E.g., `job=admin` becomes `job_admin=1` |
| **Train-Test Split** | Split data into 80% training, 20% testing | We train on one part, test on another to check real performance |
| **Standard Scaling** | Normalize numerical features to similar ranges | Some models (like Logistic Regression) work better when features are on the same scale |

In [ ]:
def preprocess_data(state: AgentState):
    """
    NODE 2: Data Preprocessing
    
    Cleans, encodes, splits, and scales the data to make it ready
    for Machine Learning models.
    """
    print("="*60)
    print("🔧 [NODE 2] Preprocessing Data...")
    print("="*60)
    
    # Re-load the data (each node can be independent)
    data_path = state["data_path"]
    df = pd.read_csv(data_path, sep=';')
    
    # Step 1: Drop any rows with missing values
    before = len(df)
    df = df.dropna()
    after = len(df)
    print(f"📌 Dropped {before - after} rows with missing values. Remaining: {after}")
    
    # Step 2: Separate Features (X) and Target (y)
    X = df.drop('y', axis=1)   # Everything except the target column
    y = df['y']                 # Only the target column
    print(f"📌 Features shape: {X.shape}, Target shape: {y.shape}")
    
    # Step 3: Encode the Target Variable
    # LabelEncoder converts: 'no' → 0, 'yes' → 1
    le_y = LabelEncoder()
    y = le_y.fit_transform(y)
    print(f"📌 Target encoded: {le_y.classes_} → {list(range(len(le_y.classes_)))}")
    
    # Step 4: One-Hot Encode Categorical Features
    # This creates new binary columns for each category
    # Example: 'marital' column with values [married, single, divorced]
    #          becomes: marital_single (0 or 1), marital_divorced (0 or 1)
    #          (drop_first=True removes one to avoid redundancy)
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    print(f"📌 Categorical columns to encode: {categorical_cols}")
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print(f"📌 After encoding, features shape: {X.shape}")
    
    # Step 5: Train-Test Split (80% training, 20% testing)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"📌 Training set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")
    
    # Step 6: Standard Scaling
    # This transforms features so they have mean=0 and std=1
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)  # Fit on training data
    X_test = scaler.transform(X_test)         # Apply same transformation to test data
    print(f"📌 Features scaled successfully!")
    
    print("\n✅ Preprocessing Complete!")
    
    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

print("Node 2 (Preprocessing) defined! ✅")

---
## Step 8: Create Node 3 — Build 3 Machine Learning Models

Now comes the exciting part! We will train **3 different ML models** and see which one performs the best.

### The 3 Models:

#### 1. Logistic Regression
- Despite the name, it's used for **classification** (not regression!)
- It draws a line (or boundary) to separate "yes" from "no"
- Simple, fast, and works well when data is linearly separable

#### 2. Decision Tree Classifier
- Think of it like a **flowchart** — it asks a series of yes/no questions to reach a decision
- Example: "Is age > 40? → Is job = admin? → Predict yes"
- Easy to understand but can "overfit" (memorize the training data too well)

#### 3. Random Forest Classifier
- Creates **many** Decision Trees (a "forest") and lets them **vote**
- The majority vote wins
- More robust and accurate than a single Decision Tree

### What is Accuracy?
**Accuracy** = (Number of correct predictions) / (Total predictions)  
Example: If a model correctly predicts 91 out of 100 → Accuracy = 91%

In [ ]:
def build_models(state: AgentState):
    """
    NODE 3: Model Building & Training
    
    Trains 3 different ML models and computes their accuracy.
    """
    print("="*60)
    print("🏗️ [NODE 3] Building & Training Models...")
    print("="*60)
    
    # Get preprocessed data from state
    X_train = state["X_train"]
    X_test = state["X_test"]
    y_train = state["y_train"]
    y_test = state["y_test"]
    
    # Define our 3 models
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42)
    }
    
    accuracies = {}  # We'll store each model's accuracy here
    
    for name, model in models.items():
        print(f"\n🚀 Training {name}...")
        
        # Step 1: Train the model on training data
        model.fit(X_train, y_train)
        
        # Step 2: Make predictions on the TEST data (data the model has never seen)
        predictions = model.predict(X_test)
        
        # Step 3: Compare predictions to actual answers
        acc = accuracy_score(y_test, predictions)
        accuracies[name] = acc
        
        print(f"   ✅ {name} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    
    print("\n" + "="*60)
    print("📊 All Model Accuracies:")
    print("="*60)
    for name, acc in accuracies.items():
        print(f"   • {name}: {acc:.4f} ({acc*100:.2f}%)")
    
    print("\n✅ Model Training Complete!")
    
    return {"model_accuracies": accuracies}

print("Node 3 (Model Building) defined! ✅")

---
## Step 9: Create Node 4 — Evaluate & Generate AI Report

### What happens here?
1. We look at all 3 model accuracies
2. We pick the **best model** (highest accuracy)
3. We use the **Groq LLM (Llama 3.3)** to write a professional summary report

### Why use an LLM for the report?
Instead of just printing numbers, the LLM generates a **human-readable executive summary** — like a real Data Science report. This shows the power of combining traditional ML with Generative AI!

In [ ]:
def evaluate_and_report(state: AgentState):
    """
    NODE 4: Evaluation & AI Report Generation
    
    Identifies the best model and uses an LLM to generate
    a polished executive summary.
    """
    print("="*60)
    print("📝 [NODE 4] Evaluating & Generating Report...")
    print("="*60)
    
    accuracies = state["model_accuracies"]
    eda_summary = state["eda_summary"]
    
    # Find the model with the highest accuracy
    best_model_name = max(accuracies, key=accuracies.get)
    best_accuracy = accuracies[best_model_name]
    
    print(f"\n🏆 BEST MODEL: {best_model_name}")
    print(f"   Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
    
    # Use the LLM to generate a professional report
    try:
        llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile")
        
        prompt = f"""
        You are a Data Science Assistant. Write a short, professional executive summary 
        based on the following ML pipeline results:
        
        EDA Highlights:
        {eda_summary}
        
        Model Accuracies:
        {accuracies}
        
        Best Model:
        {best_model_name} with accuracy {best_accuracy:.4f}
        
        Write a concise, bulleted summary explaining:
        - What the data looks like
        - Which model performed best and why that matters
        - Any recommendations
        """
        
        response = llm.invoke(prompt)
        final_report = response.content
        
        print("\n" + "="*60)
        print("🤖 AI-Generated Final Report:")
        print("="*60)
        print(final_report)
        
    except Exception as e:
        print(f"\n⚠️ Could not generate LLM report: {e}")
        print("Generating fallback report...")
        final_report = (
            f"Best model: {best_model_name} with accuracy {best_accuracy:.4f}.\n"
            f"All accuracies: {accuracies}"
        )
        print(f"\n{final_report}")
    
    return {"best_model_name": best_model_name, "final_report": final_report}

print("Node 4 (Evaluation) defined! ✅")

---
## Step 10: Build the LangGraph Workflow

### How LangGraph works:

We create a **graph** where:
- Each **node** is a function (our 4 steps above)
- Each **edge** connects one node to the next
- `START` is where the graph begins
- `END` is where it finishes

### Our Flow:
```
START → perform_eda → preprocess_data → build_models → evaluate_and_report → END
```

This is a **linear pipeline** — one step after another. LangGraph also supports branching and cycles for more complex workflows, but a linear flow is perfect for our use case.

In [ ]:
# --- Build the LangGraph Workflow ---

# Step 1: Create a graph with our defined state
workflow = StateGraph(AgentState)

# Step 2: Add all 4 nodes to the graph
workflow.add_node("perform_eda", perform_eda)
workflow.add_node("preprocess_data", preprocess_data)
workflow.add_node("build_models", build_models)
workflow.add_node("evaluate_and_report", evaluate_and_report)

# Step 3: Add edges to define the execution order
workflow.add_edge(START, "perform_eda")              # Start → EDA
workflow.add_edge("perform_eda", "preprocess_data")   # EDA → Preprocessing
workflow.add_edge("preprocess_data", "build_models")  # Preprocessing → Model Building
workflow.add_edge("build_models", "evaluate_and_report")  # Model Building → Evaluation
workflow.add_edge("evaluate_and_report", END)         # Evaluation → End

# Step 4: Compile the graph into a runnable application
app = workflow.compile()

print("LangGraph Workflow compiled successfully! ✅")
print("\nGraph flow: START → EDA → Preprocess → Build Models → Evaluate → END")

---
## Step 11: 🚀 Run the Agent!

This is the moment of truth! We invoke (start) the graph with an **initial state** containing just the path to our CSV file.

The agent will then autonomously:
1. Explore the data
2. Clean and preprocess it
3. Train 3 models
4. Pick the best one and write a report

**All in one call!** 🎉

In [ ]:
# Define the starting input — just the file path
initial_state = {"data_path": DATA_PATH}

print("🤖 Initiating LangGraph Data Science Agent Workflow...\n")

# Run the entire pipeline!
result = app.invoke(initial_state)

print("\n" + "="*60)
print("🎉 AGENT WORKFLOW COMPLETE!")
print("="*60)

---
## Step 12: View Final Results

The `result` variable contains the **final state** after all 4 nodes have run.  
Let's inspect the key outputs.

In [ ]:
# Print the winning model
print("🏆 Best Model:", result["best_model_name"])
print()

# Print all model accuracies
print("📊 All Model Accuracies:")
for model_name, accuracy in result["model_accuracies"].items():
    marker = " ← 🥇 BEST" if model_name == result["best_model_name"] else ""
    print(f"   • {model_name}: {accuracy:.4f} ({accuracy*100:.2f}%){marker}")

print()
print("📝 AI-Generated Report:")
print(result["final_report"])

---
## 🧠 Summary: What We Learned

### Concepts Covered:

| Concept | What it is |
|---------|------------|
| **LangChain** | Framework to connect LLMs with tools and data |
| **LangGraph** | Builds workflows as graphs (nodes + edges) |
| **StateGraph** | A LangGraph graph where data passes through a shared state |
| **Nodes** | Individual functions/steps in the pipeline |
| **Edges** | Connections defining execution order |
| **TypedDict** | Python type hints for structured dictionaries |
| **EDA** | Exploring and understanding your dataset |
| **Preprocessing** | Cleaning and transforming data for ML |
| **Label Encoding** | Converting text labels to numbers |
| **One-Hot Encoding** | Converting categories to binary columns |
| **Standard Scaling** | Normalizing features to similar ranges |
| **Train-Test Split** | Dividing data for training and evaluation |
| **Logistic Regression** | Linear classifier for binary outcomes |
| **Decision Tree** | Flowchart-based classifier |
| **Random Forest** | Ensemble of many Decision Trees |
| **Accuracy** | % of correct predictions |
| **Groq + Llama 3.3** | Fast LLM API for generating reports |

### The Power of Agentic AI:
We combined **traditional ML** (scikit-learn) with **Generative AI** (LLM via LangChain) in a **structured graph workflow** (LangGraph). This is the foundation of building real-world AI agents!

---
**Made with ❤️ using LangChain + LangGraph**